In [1]:
# --- headless execution harness (added by run_experiments.py) ---
import os as _h_os
import matplotlib as _h_mpl
_h_mpl.use('Agg')
import matplotlib.pyplot as _h_plt
_OUT = 'C:\\Users\\ASUS\\group_sgu26_deep_learning\\BTL03\\output\\5. Others\\2_obj_detection_pytorch'
_h_os.makedirs(_OUT, exist_ok=True)
_h_n = [0]
_h_show = _h_plt.show
def _saving_show(*args, **kwargs):
    _figs = [_h_plt.figure(n) for n in _h_plt.get_fignums()]
    for _f in _figs:
        _h_n[0] += 1
        _f.savefig(_h_os.path.join(_OUT, 'fig_%02d.png' % _h_n[0]), dpi=110, bbox_inches='tight')
    _h_show(*args, **kwargs)
    for _f in _figs:
        _h_plt.close(_f)
_h_plt.show = _saving_show
print('[harness] figures -> ' + _OUT)


[harness] figures -> C:\Users\ASUS\group_sgu26_deep_learning\BTL03\output\5. Others\2_obj_detection_pytorch


In [2]:
# Demo (PyTorch)
# based on https://github.com/HTuennermann/Vehicle-Detection-and-Tracking

# Overview
- Simple fully-convolutional object detector (PyTorch)
- Input: RGB image `(3, 64, 64)` khi train; ảnh kích thước bất kỳ `(3, H, W)` khi suy luận heatmap
- Output: cars' locations (heatmap + bounding boxes)

# Import libraries

In [3]:
import glob
import numpy as np
from skimage.io import imread
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


# (1) Data

#### Car images: https://drive.google.com/file/d/1rqitUFrFhB84tFVtg1Ad1TVz6coCWpNZ/view?usp=sharing

In [4]:
cars = glob.glob('./data/obj_detection/vehicles/*/*.png')
non_cars = glob.glob('./data/obj_detection/non-vehicles/*/*.png')

Y = np.concatenate([np.ones(len(cars), dtype=np.float32), -np.ones(len(non_cars), dtype=np.float32)])
X = []
for name in cars:
    img = imread(name)
    X.append(img[:, :, :3])  # đảm bảo đúng 3 kênh RGB nếu PNG có kênh alpha
for name in non_cars:
    img = imread(name)
    X.append(img[:, :, :3])
X = np.array(X, dtype=np.float32)

# Chuyển NHWC (N, 64, 64, 3) -> NCHW (N, 3, 64, 64) cho PyTorch Conv2d
X = np.transpose(X, (0, 3, 1, 2)).copy()
print('Loaded X:', X.shape, '| Y:', Y.shape)

Loaded X: (13745, 3, 64, 64) | Y: (13745,)


In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30, random_state=42)

print('X_train shape:', X_train.shape)
print(X_train.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')

X_train shape: (9621, 3, 64, 64)
9621 train samples
4124 test samples


# (2) Declare model

In [6]:
class CarDetectorFCN(nn.Module):
    """Mạng tích chập hoàn toàn (FCN): chấp nhận cả ảnh 64x64 khi train lẫn ảnh kích thước bất kỳ (H, W) khi tạo heatmap."""
    def __init__(self, flatten_output=False):
        super().__init__()
        self.flatten_output = flatten_output
        self.conv1 = nn.Conv2d(3, 10, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(10, 10, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=8, stride=8)
        self.dropout1 = nn.Dropout(0.25)
        self.dense1 = nn.Conv2d(10, 128, kernel_size=8, padding=0)
        self.dropout2 = nn.Dropout(0.5)
        self.dense2 = nn.Conv2d(128, 1, kernel_size=1, padding=0)

    def forward(self, x):
        # Tương đương Lambda(lambda x: x / 127.5 - 1.)
        x = x / 127.5 - 1.0
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.dropout1(self.pool(x))
        x = self.dropout2(F.relu(self.dense1(x)))
        x = torch.tanh(self.dense2(x))  # (B, 1, H_out, W_out)
        if self.flatten_output:
            return x.flatten(start_dim=1)  # (B, 1) cho ảnh 64x64
        return x

model = CarDetectorFCN(flatten_output=True).to(device)
print(model)
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')  # Khớp chính xác 83,367 params của bản Keras

CarDetectorFCN(
  (conv1): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=8, stride=8, padding=0, dilation=1, ceil_mode=False)
  (dropout1): Dropout(p=0.25, inplace=False)
  (dense1): Conv2d(10, 128, kernel_size=(8, 8), stride=(1, 1))
  (dropout2): Dropout(p=0.5, inplace=False)
  (dense2): Conv2d(128, 1, kernel_size=(1, 1), stride=(1, 1))
)
Total params: 83,367


In [7]:
try:
    from torchinfo import summary
    print(summary(model, input_size=(1, 3, 64, 64), device=device))
except ImportError:
    print('Cài đặt torchinfo nếu muốn xem bảng summary chi tiết: pip install torchinfo')

Layer (type:depth-idx)                   Output Shape              Param #
CarDetectorFCN                           [1, 1]                    --
├─Conv2d: 1-1                            [1, 10, 64, 64]           280
├─Conv2d: 1-2                            [1, 10, 64, 64]           910
├─MaxPool2d: 1-3                         [1, 10, 8, 8]             --
├─Dropout: 1-4                           [1, 10, 8, 8]             --
├─Conv2d: 1-5                            [1, 128, 1, 1]            82,048
├─Dropout: 1-6                           [1, 128, 1, 1]            --
├─Conv2d: 1-7                            [1, 1, 1, 1]              129
Total params: 83,367
Trainable params: 83,367
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 4.96
Input size (MB): 0.05
Forward/backward pass size (MB): 0.66
Params size (MB): 0.33
Estimated Total Size (MB): 1.04


# (3) Train defined model

In [8]:
def evaluate_detector(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            preds = model(bx)
            loss = criterion(preds, by)
            total_loss += loss.item() * bx.size(0)
            # Dự đoán > 0 là xe (+1), <= 0 là không phải xe (-1)
            correct += ((preds >= 0.0) == (by >= 0.0)).sum().item()
            total += bx.size(0)
    return total_loss / total, correct / total


criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

batch_size = 128
epochs = 5

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train).unsqueeze(1)),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test), torch.from_numpy(Y_test).unsqueeze(1)),
    batch_size=batch_size, shuffle=False
)

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        preds = model(bx)
        loss = criterion(preds, by)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * bx.size(0)
        correct += ((preds >= 0.0) == (by >= 0.0)).sum().item()
        total += bx.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate_detector(model, test_loader, criterion, device)
    print(f'Epoch {epoch}/{epochs} - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')

test_loss, test_acc = evaluate_detector(model, test_loader, criterion, device)
print('Test score (MSE):', test_loss)
print('Test accuracy:', test_acc)

torch.save(model.state_dict(), 'localize.pt')
print('Saved weights to localize.pt')

Epoch 1/5 - loss: 0.5079 - acc: 0.8279 - val_loss: 0.2505 - val_acc: 0.9302


Epoch 2/5 - loss: 0.2392 - acc: 0.9279 - val_loss: 0.2198 - val_acc: 0.9287


Epoch 3/5 - loss: 0.1885 - acc: 0.9395 - val_loss: 0.1496 - val_acc: 0.9520


Epoch 4/5 - loss: 0.1524 - acc: 0.9537 - val_loss: 0.1154 - val_acc: 0.9641


Epoch 5/5 - loss: 0.1270 - acc: 0.9614 - val_loss: 0.0988 - val_acc: 0.9692
Test score (MSE): 0.09881522436377843
Test accuracy: 0.9692046556741029
Saved weights to localize.pt


# (4) Evaluate trained model (Heatmap & Car Localization)

In [9]:
# Nạp trọng số vào mạng FCN (flatten_output=False để xuất heatmap 2D cho ảnh kích thước bất kỳ)
heatmodel = CarDetectorFCN(flatten_output=False).to(device)
heatmodel.load_state_dict(torch.load('localize.pt', map_location=device))
heatmodel.eval()

CarDetectorFCN(
  (conv1): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=8, stride=8, padding=0, dilation=1, ceil_mode=False)
  (dropout1): Dropout(p=0.25, inplace=False)
  (dense1): Conv2d(10, 128, kernel_size=(8, 8), stride=(1, 1))
  (dropout2): Dropout(p=0.5, inplace=False)
  (dense2): Conv2d(128, 1, kernel_size=(1, 1), stride=(1, 1))
)

In [10]:
def locate(image_path='test.jpg', threshold=0.99):
    data = imread(image_path)[:, :, :3].copy()  # (H, W, 3) uint8
    H, W, _ = data.shape

    inp = np.transpose(data.astype(np.float32), (2, 0, 1))  # (3, H, W)
    inp = torch.from_numpy(np.expand_dims(inp, axis=0)).to(device)

    with torch.no_grad():
        heatmap = heatmodel(inp).cpu().numpy()[0, 0, :, :]  # (H_hm, W_hm)

    sel_heatmap = heatmap > threshold
    xx, yy = np.meshgrid(np.arange(heatmap.shape[1]), np.arange(heatmap.shape[0]))
    xs = xx[sel_heatmap]
    ys = yy[sel_heatmap]

    for i, j in zip(xs, ys):
        y1, y2 = j * 8, min(j * 8 + 64, H - 1)
        x1, x2 = i * 8, min(i * 8 + 64, W - 1)
        data[y1:y2 + 1, x1] = (0, 255, 0)
        data[y1:y2 + 1, x2] = (0, 255, 0)
        data[y1, x1:x2 + 1] = (0, 255, 0)
        data[y2, x1:x2 + 1] = (0, 255, 0)

    return heatmap, sel_heatmap, data


hm, sel_hm, annotated = locate('test.jpg', threshold=0.99)

plt.figure(figsize=(10, 20))

plt.subplot(3, 1, 1)
plt.title('Heatmap')
plt.imshow(hm)

plt.subplot(3, 1, 2)
plt.title('Car Area')
plt.imshow(sel_hm)

plt.subplot(3, 1, 3)
plt.title('Detected Cars')
plt.imshow(annotated)
plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_17968\3332401834.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)
